In [1]:
import sys
import os
import time
from dotenv import load_dotenv
import pandas as pd
from datasets import Dataset

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, AnswerRelevancy
from ragas.run_config import RunConfig

_PROJECT_ROOT = os.path.abspath("..")
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
load_dotenv(os.path.join(_PROJECT_ROOT, ".env"))

from src import config
from src.graph import app
from src.nodes import retriever, format_docs
from src.main import setup_observability

# Phoenix UI — open the printed URL and watch Traces while cells run
setup_observability()

# Groq only allows n=1; default answer_relevancy uses strictness=3
answer_relevancy = AnswerRelevancy(strictness=1)
RAGAS_RUN_CONFIG = RunConfig(max_workers=1, max_retries=5, timeout=120)
MAX_CONTEXT_CHARS = 3000

eval_llm = ChatGroq(model=config.LLM_MODEL, temperature=0)
eval_embeddings = HuggingFaceEmbeddings(
    model_name=config.EMBEDDING_MODEL,
    model_kwargs={"device": config.EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)

naive_prompt = PromptTemplate.from_template(
    "Answer the question using the context. If you don't know, say so.\n"
    "Context: {context}\nQuestion: {question}\nAnswer:"
)
naive_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | naive_prompt
    | eval_llm
    | StrOutputParser()
)


def truncate_contexts(contexts: list[str]) -> list[str]:
    return [
        text[:MAX_CONTEXT_CHARS] + "..." if len(text) > MAX_CONTEXT_CHARS else text
        for text in contexts
    ]


def run_ragas(dataset_dict: dict, label: str):
    print(f"\n--- Running Ragas on {label} ---")
    result = evaluate(
        Dataset.from_dict(dataset_dict),
        metrics=[faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_emb,
        run_config=RAGAS_RUN_CONFIG,
        allow_nest_asyncio=True,
    )
    df = result.to_pandas()
    question_col = "user_input" if "user_input" in df.columns else "question"
    print(df[[question_col, "faithfulness", "answer_relevancy"]])
    print("Mean scores:", df[["faithfulness", "answer_relevancy"]].mean(numeric_only=True).to_dict())
    return result, df


print("Setup complete.")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/var/folders/gc/5sc3n9hs70bgbch5c1skcqbr0000gp/T/ipykernel_68375/534361050.py:17: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Users/ramunalla/Personal/sentinel-RAG/src/nodes.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
/Users/ramunalla/Personal/sentinel-RAG/src/nodes.py:44: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip instal

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/contextlib.py:148: SAWarning: Skipped unsupported reflection of expression-based index ix_spans_session_id
  next(self.gen)
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: sentinel-rag
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

👁️ Observability Dashboard running at: http://localhost:6006/


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup complete.


/var/folders/gc/5sc3n9hs70bgbch5c1skcqbr0000gp/T/ipykernel_68375/534361050.py:44: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(eval_llm)
/var/folders/gc/5sc3n9hs70bgbch5c1skcqbr0000gp/T/ipykernel_68375/534361050.py:45: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(eval_embeddings)


In [2]:
# --- Part A: Quick 3-question SentinelRAG smoke test ---
from src.main import process_query
from src.cache import clear_cache

eval_questions = [
    "What is the concept of Self-RAG?",
    "How does Self-RAG handle hallucinations?",
    "What is the weather in Tokyo?",
]
ground_truths = [
    "Self-RAG is a framework that retrieves relevant passages on-demand, and uses self-reflection to critique and select the best outputs.",
    "It uses a self-reflection mechanism with critic models to evaluate if the generated text is grounded in the retrieved facts.",
    "I do not have real-time weather information in my local database, but a web search shows the current weather in Tokyo.",
]

clear_cache()
agentic_answers, retrieved_contexts = [], []
for q in eval_questions:
    result = process_query(q, use_cache=False)
    agentic_answers.append(result["answer"])
    retrieved_contexts.append(truncate_contexts(result["contexts"] or ["No context retrieved."]))

_, df_quick = run_ragas(
    {
        "question": eval_questions,
        "answer": agentic_answers,
        "ground_truth": ground_truths,
        "contexts": retrieved_contexts,
    },
    label="SentinelRAG (quick)",
)

--- CACHE CLEARED (0 entries removed) ---

[USER QUERY]: What is the concept of Self-RAG?

--- INITIATING AGENTIC WORKFLOW ---
---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Relevant
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openinference/instrumentation/_spans.py:43: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  masked_value = self._self_config.mask(key, value)


---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Hallucination).
---EDGE: CHECK IF ANSWER RESOLVES QUERY---
  - DECISION: Answer is Useful and Resolves Query.

[FINAL ANSWER FROM AGENT]:
Self-RAG is a concept that involves a retriever model retrieving relevant passages, then concurrently processing and evaluating them to generate task outputs. It also critiques its own output and chooses the best one based on factuality and quality, allowing for easier fact verification.

🐢 [LATENCY]: 2.6754 seconds 🐢

[USER QUERY]: How does Self-RAG handle hallucinations?

--- INITIATING AGENTIC WORKFLOW ---
---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Irrelevant (REJECTED)
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---
---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Halluc

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

                                 user_input  faithfulness  answer_relevancy
0          What is the concept of Self-RAG?      0.833333          0.929086
1  How does Self-RAG handle hallucinations?      1.000000          0.977344
2             What is the weather in Tokyo?      0.400000          0.958389
Mean scores: {'faithfulness': 0.7444444444444445, 'answer_relevancy': 0.9549400518630814}


## Part B: Full benchmark — Naive RAG vs SentinelRAG (20 QA pairs)

Open the **Phoenix dashboard URL** from the setup cell while this runs to inspect traces for every LLM call.

**Note:** This takes a long time (~20+ min) — 20 questions × 2 pipelines + Ragas. The last 5 questions need `TAVILY_API_KEY` in `.env`.

In [3]:
# --- Part B: 20 QA pairs — Naive RAG vs SentinelRAG ---
qa_pairs = [
    ("What is the main purpose of the Self-RAG framework?", "To enhance the quality and factuality of LLM generation through self-reflection and on-demand retrieval."),
    ("How does Self-RAG decide when to retrieve documents?", "It uses a critic model to predict a retrieval token, determining if retrieval is necessary based on the context."),
    ("What are the two main types of reflection tokens used in Self-RAG?", "Retrieve tokens (to decide if retrieval is needed) and Critic tokens (to evaluate the generated output)."),
    ("Does Self-RAG retrieve documents for every single query?", "No, it retrieves on-demand only when the model determines that factual context is required."),
    ("How does Self-RAG handle hallucinated context?", "It evaluates the retrieved context and the generated response, rejecting outputs that are not supported by the evidence."),
    ("What baseline models does Self-RAG compare itself against?", "It typically compares against standard/naive RAG and non-retrieval LLMs like ChatGPT or standard Llama."),
    ("What does the 'critic' do in the Self-RAG training phase?", "The critic model is trained to generate reflection tokens that evaluate the quality and groundedness of responses."),
    ("Is Self-RAG training an open-source or closed-source process?", "Self-RAG involves training open-source models (like Llama 2) to generate reflection tokens."),
    ("What is the 'utility' score in Self-RAG?", "A score used to rank and select the best candidate response based on how well it answers the query and uses the context."),
    ("Does Self-RAG reduce API costs compared to standard RAG?", "Yes, by skipping retrieval for conversational or simple queries, it can save on retrieval and embedding costs."),
    ("What dataset was used to train the critic model in Self-RAG?", "It uses synthesized data generated by GPT-4 to train the smaller critic model."),
    ("Can Self-RAG be applied to tasks other than QA?", "Yes, it can be applied to reasoning, summarization, and instruction-following tasks."),
    ("How does the generation phase of Self-RAG work?", "It generates multiple candidate segments, critiques them using reflection tokens, and selects the best one."),
    ("What happens if retrieved documents are entirely irrelevant in Self-RAG?", "The model's critique tokens will flag it as unsupported, and it may choose to ignore the context or output a failure response."),
    ("Why is standard RAG sometimes harmful?", "Standard RAG might blindly use retrieved context even if it is irrelevant or incorrect, leading to worse answers."),
    ("What was the closing price of Apple stock yesterday?", "Apple's stock price varies daily, but standard local vector DBs would not have this real-time data, requiring a web search."),
    ("Who won the Super Bowl in 2024?", "The Kansas City Chiefs won the Super Bowl in 2024."),
    ("What is the current weather forecast for New York City?", "Current weather conditions require real-time web retrieval to provide an accurate forecast."),
    ("Who is the current CEO of OpenAI?", "Sam Altman is the current CEO of OpenAI."),
    ("What are the top news headlines today?", "Top news headlines require live web search to retrieve the most recent events."),
]

questions = [q for q, _ in qa_pairs]
ground_truths = [gt for _, gt in qa_pairs]

naive_answers, sentinel_answers = [], []
naive_contexts, sentinel_contexts = [], []

print(f"Starting full benchmark on {len(questions)} questions...")
print("Watch traces in Phoenix while this runs.\n")

for i, q in enumerate(questions):
    print(f"Processing Q{i + 1}/{len(questions)}: {q}")

    naive_docs = retriever.invoke(q)
    naive_answers.append(naive_rag_chain.invoke(q))
    naive_contexts.append(truncate_contexts([doc.page_content for doc in naive_docs]))
    time.sleep(1)

    agent_state = app.invoke({"question": q, "retries": 0, "web_search_count": 0})
    sentinel_answers.append(agent_state.get("generation", ""))
    docs = agent_state.get("documents", [])
    sentinel_contexts.append(
        truncate_contexts([doc.page_content for doc in docs] if docs else ["No context retrieved."])
    )
    time.sleep(2)

naive_data = {
    "question": questions,
    "answer": naive_answers,
    "contexts": naive_contexts,
    "ground_truth": ground_truths,
}
sentinel_data = {
    "question": questions,
    "answer": sentinel_answers,
    "contexts": sentinel_contexts,
    "ground_truth": ground_truths,
}

_, df_naive = run_ragas(naive_data, label="Naive RAG")
_, df_sentinel = run_ragas(sentinel_data, label="SentinelRAG")

print("\n🏆 FINAL COMPARISON (mean scores) 🏆")
comparison = pd.DataFrame(
    {
        "Metric": ["Faithfulness", "Answer Relevancy"],
        "Naive RAG": [
            df_naive["faithfulness"].mean(),
            df_naive["answer_relevancy"].mean(),
        ],
        "SentinelRAG": [
            df_sentinel["faithfulness"].mean(),
            df_sentinel["answer_relevancy"].mean(),
        ],
    }
)
print(comparison.to_string(index=False))

Starting full benchmark on 20 questions...
Watch traces in Phoenix while this runs.

Processing Q1/20: What is the main purpose of the Self-RAG framework?


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/openinference/instrumentation/_spans.py:43: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  masked_value = self._self_config.mask(key, value)


---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Relevant
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---
---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Hallucination).
---EDGE: CHECK IF ANSWER RESOLVES QUERY---
  - DECISION: Answer is Useful and Resolves Query.
Processing Q2/20: How does Self-RAG decide when to retrieve documents?
---NODE: RETRIEVE FROM VECTOR DB---
---NODE: GRADE DOCUMENT RELEVANCE---
  - GRADE: Document Relevant
  - GRADE: Document Relevant
  - GRADE: Document Relevant
---EDGE: EVALUATE RETRIEVAL RESULTS---
  - DECISION: Documents relevant. Routing to Generate.
---NODE: GENERATE ANSWER---
---EDGE: CHECK FOR HALLUCINATIONS---
  - DECISION: Answer is Grounded (No Hallucination).
---EDGE: CHECK IF ANSWER RESOLVES QUERY---
  - DECISION: Answer is Useful and Resolves Quer

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqm5p3mxezj9d0d71brp6a03` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98973, Requested 3185. Please try again in 31m4.512s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}